In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/ex1_createtable_predictions.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/finetuning_ready/dev.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/finetuning_ready/train.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/finetuning_ready/test.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/splits/train_en.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/splits/test_en.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/splits/split_info.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/splits/dev_en.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/hindi_translated/test_hi.json
/kaggle/input/datasets/aniruddhabhagat/hindidb-finetun

In [2]:
import os
print(os.listdir('/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data'))

['HindiDB_for_kaggle', 'ex1_createtable_predictions.json']


In [3]:
!pip install "transformers==4.43.4" -q
!pip install "peft==0.11.1" -q
!pip install "bitsandbytes==0.49.2" -q
!pip install "accelerate==0.30.0" -q
!pip install "trl==0.8.6" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 103.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 111.3 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible

In [4]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

import transformers, peft, trl, accelerate, bitsandbytes
print(f"transformers: {transformers.__version__}, peft: {peft.__version__}, trl: {trl.__version__}")

CUDA available: True
GPU: Tesla T4
transformers: 4.43.4, peft: 0.11.1, trl: 0.8.6


In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ HuggingFace login successful")

✅ HuggingFace login successful


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "Qwen/Qwen2.5-Coder-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
GPU memory used: 2.08 GB


In [7]:
import json
from datasets import Dataset

data_base = '/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle'

def load_as_hf_dataset(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    formatted = []
    for row in data:
        full_text = row['prompt'] + row['completion'] + tokenizer.eos_token
        formatted.append({'text': full_text, 'prompt': row['prompt'], 'completion': row['completion']})
    return Dataset.from_list(formatted)

# Load SERIALIZED format this time (not CREATE TABLE)
train_dataset_b = load_as_hf_dataset(f'{data_base}/finetuning_ready_serialized/train.json')
dev_dataset_b = load_as_hf_dataset(f'{data_base}/finetuning_ready_serialized/dev.json')

print(f"Train (Serialized): {len(train_dataset_b)} examples")
print(f"Dev (Serialized):   {len(dev_dataset_b)} examples")

Train (Serialized): 5232 examples
Dev (Serialized):   528 examples


In [8]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os

os.makedirs('/kaggle/working/models/qwen_ex1_serialized', exist_ok=True)

training_args = TrainingArguments(
    output_dir='/kaggle/working/models/qwen_ex1_serialized',
    max_steps=350,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=14,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none",       # no wandb needed for Run B, saves overhead
    run_name="qwen-ex1-serialized",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_b,
    eval_dataset=dev_dataset_b,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

print("✅ Trainer ready for EX1 Run B")

2026-06-22 12:56:10.429611: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782132970.606113      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782132970.651778      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782132971.078748      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782132971.078786      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782132971.078791      58 computation_placer.cc:177] computation placer alr

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


✅ Trainer ready for EX1 Run B


In [12]:
# Confirm checkpoint-25 has the best eval_loss
import json
state_path = f'{checkpoint_dir}/checkpoint-25/trainer_state.json'
with open(state_path) as f:
    state = json.load(f)

print("All eval entries:")
for entry in state['log_history']:
    if 'eval_loss' in entry:
        print(f"  Step {entry['step']}: eval_loss = {entry['eval_loss']:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/models/qwen_ex1_serialized/checkpoint-25/trainer_state.json'

In [10]:
# Recreate trainer with max_steps=25 only — we just need that one checkpoint
training_args = TrainingArguments(
    output_dir='/kaggle/working/models/qwen_ex1_serialized',
    max_steps=25,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_b,
    eval_dataset=dev_dataset_b,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

print("✅ Trainer ready — 25 steps only")

Map:   0%|          | 0/5232 [00:00<?, ? examples/s]

Map:   0%|          | 0/528 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


✅ Trainer ready — 25 steps only


In [13]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being ke

Step,Training Loss,Validation Loss
25,0.988900,1.181957


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


TrainOutput(global_step=25, training_loss=1.1815225028991698, metrics={'train_runtime': 438.7644, 'train_samples_per_second': 0.912, 'train_steps_per_second': 0.057, 'total_flos': 1999632917741568.0, 'train_loss': 1.1815225028991698, 'epoch': 0.0764525993883792})

In [11]:
# Verify checkpoint saved
import os
checkpoint_dir = '/kaggle/working/models/qwen_ex1_serialized'
print(os.listdir(checkpoint_dir))

[]


In [14]:
# Load test set — serialized format this time
import json

test_path = '/kaggle/input/datasets/aniruddhabhagat/hindidb-finetuning-data/HindiDB_for_kaggle/finetuning_ready_serialized/test.json'
with open(test_path, encoding='utf-8') as f:
    test_data_b = json.load(f)

print(f"Test set size: {len(test_data_b)}")

Test set size: 381


In [15]:
# Generation function — same as Run A with FIM token stripping
def generate_sql(prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )
    
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_sql = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    for special_str in ['<|fim_middle|>', '<|fim_prefix|>', '<|fim_suffix|>', '<|endoftext|>', '<|im_end|>']:
        generated_sql = generated_sql.replace(special_str, '')
    
    generated_sql = generated_sql.split('\n')[0].strip()
    return generated_sql

In [16]:
# Run generation across all 381 test examples immediately
import time

predictions_b = []
start_time = time.time()

for i, row in enumerate(test_data_b):
    pred_sql = generate_sql(row['prompt'])
    predictions_b.append({
        'db_id': row['db_id'],
        'prediction': pred_sql,
        'gold': row['completion']
    })
    
    if i % 25 == 0:
        elapsed = time.time() - start_time
        print(f"Progress: {i+1}/{len(test_data_b)} — {elapsed:.0f}s elapsed")

print(f"\n✅ Done — {len(predictions_b)} predictions generated")

Progress: 1/381 — 23s elapsed
Progress: 26/381 — 394s elapsed
Progress: 51/381 — 767s elapsed
Progress: 76/381 — 1083s elapsed
Progress: 101/381 — 1484s elapsed
Progress: 126/381 — 1860s elapsed
Progress: 151/381 — 2136s elapsed
Progress: 176/381 — 2511s elapsed
Progress: 201/381 — 2886s elapsed
Progress: 226/381 — 3272s elapsed
Progress: 251/381 — 3684s elapsed
Progress: 276/381 — 4020s elapsed
Progress: 301/381 — 4290s elapsed
Progress: 326/381 — 4648s elapsed
Progress: 351/381 — 4987s elapsed
Progress: 376/381 — 5388s elapsed

✅ Done — 381 predictions generated


In [17]:
# Save immediately — don't risk losing this
os.makedirs('/kaggle/working/results', exist_ok=True)
with open('/kaggle/working/results/ex1_serialized_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(predictions_b, f, ensure_ascii=False, indent=2)

print("✅ Predictions saved to /kaggle/working/results/")

✅ Predictions saved to /kaggle/working/results/
